# Dataset 2: Mandi Master Data Profiling & Rescue Notebook

### Step 1: Raw Data Profiling and Primary Key Deduplication

**Problem Strategy**:
1. Mandi Master serves as the primary Dimension Table (`DIM_MANDI`) in our system. Duplicate `mandi_id` entries in a dimension table cause multi-counting (fan-out effect) during SQL `JOIN` operations against arrival and transport fact tables.
2. We inspect raw duplicate primary keys using `duplicated()` and deduplicate on `mandi_id` to enforce strict primary key uniqueness.


In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Load Raw Datset
raw_master_path = "../data/raw/track3_mandi_master.csv"
df_raw = pd.read_csv(raw_master_path)

In [13]:
df.head(10)

,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
0,MANDI001,Hyderabad Mandi,ludhiana,Punjab,Private,11.0
1,MANDI006,Kochi Mandi,Amritsar,Punjab,Direct,34.0
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,NaN
3,MANDI046,Baranagar Market,NaN,Uttar Pradesh,Private,13.0
4,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
5,MANDI055,Chandigarh APMC,Bareilly,Uttar Pradesh,apmc,12.0
6,MANDI034,Ludhiana Grain Market,Ambala,Haryana,NaN,11.0
7,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,Private,39.0
8,MANDI013,Aurangabad Market,Patiala,Punjab,Private,25.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,NaN


In [3]:
# Check Raw Dimensions & Duplicates
initial_rows = len(df_raw)
duplicate_ids_before = df_raw['mandi_id'].duplicated().sum()

In [5]:
#  BEFORE Deduplication Scan
print("Total Raw Rows Loaded:", initial_rows)
print("Duplicate mandi_id entries found:", duplicate_ids_before)

Total Raw Rows Loaded: 60
Duplicate mandi_id entries found: 3


In [7]:
# Duplicate mandi_id entries found in the raw dataset
print(df_raw[df_raw['mandi_id'].duplicated(keep=False)])

    mandi_id         mandi_name  district    state mandi_type  \
0   MANDI001    Hyderabad Mandi  ludhiana   Punjab    Private   
1   MANDI006        Kochi Mandi  Amritsar   Punjab     Direct   
9   MANDI001    Hyderabad Mandi  ludhiana   Punjab    Private   
23  MANDI006        Kochi Mandi  Amritsar   Punjab     Direct   
29  MANDI031  Nangloi Jat Mandi     Sirsa  Haryana       APMC   
52  MANDI031  Nangloi Jat Mandi     Sirsa  Haryana       APMC   

    total_area_acres  
0               11.0  
1               34.0  
9               11.0  
23              34.0  
29              22.0  
52              22.0  


In [8]:
df = df_raw.drop_duplicates(subset=['mandi_id']).copy()

In [9]:
# Check Clean Dimensions
duplicate_ids_after = df['mandi_id'].duplicated().sum()
deduplicated_rows = len(df)


In [12]:
print("AFTER Deduplication Verification: ")
print()
print("Total Unique Mandi Master Rows Remaining:", deduplicated_rows)
print("Duplicate mandi_id entries remaining:", duplicate_ids_after)


AFTER Deduplication Verification: 

Total Unique Mandi Master Rows Remaining: 57
Duplicate mandi_id entries remaining: 0


### Step 2: Standardizing Mandi Type Casing and Imputing Missing Categories

**Problem Strategy**:
1. Raw `mandi_type` entries contain inconsistent casing (`apmc` vs `APMC`, `PRIVATE` vs `Private`). We normalize them into 3 standard enums: `APMC`, `Private`, `Direct`.
2. Missing `mandi_type` entries are imputed with the most frequent mandi category (`APMC`) so zero records are lost.


In [16]:
print("BEFORE Mandi Type Distribution :- ")
print(df['mandi_type'].value_counts(dropna=False))


BEFORE Mandi Type Distribution :- 
mandi_type
APMC       14
NaN        11
Private     9
Direct      9
PRIVATE     8
apmc        6
Name: count, dtype: int64


In [17]:
# Mandi Type Mapping
mandi_type_map = {
    'APMC': 'APMC', 'apmc': 'APMC',
    'Private': 'Private', 'PRIVATE': 'Private',
    'Direct': 'Direct'
}

In [22]:
# Apply mapping and fill missing NaN values with 'APMC'
df['clean_mandi_type'] = df['mandi_type'].apply(lambda x: mandi_type_map.get(str(x).strip(), str(x).strip()) if pd.notna(x) else 'APMC')

In [23]:
print("AFTER Mandi Type Standardisation Verification:- ")
print()
print(df['clean_mandi_type'].value_counts(dropna=False))

AFTER Mandi Type Standardisation Verification:- 

clean_mandi_type
APMC       31
Private    17
Direct      9
Name: count, dtype: int64


### Step 3: Imputing Missing Geographic Location (District and State)

**Problem Strategy**:
1. 4 mandi records (`MANDI046`, `MANDI032`, `MANDI017`, `MANDI039`) have missing `district` and `state` entries.
2. Dropping master mandis would cause orphan records in arrival and transport datasets. We impute missing locations using a market location lookup dictionary.


In [35]:
# Check missing district & state count
print("BEFORE Geographic Imputation Scan:- ")
print("Missing district entries before:", df['district'].isnull().sum())
print("Missing state entries before:", df['state'].isnull().sum())

BEFORE Geographic Imputation Scan:- 
Missing district entries before: 4
Missing state entries before: 4


In [40]:
# Locatiobn mapping
location_lookup = {
    # 4 mandis with missing district
    'MANDI046': {'district': 'Kolkata', 'state': 'West Bengal'},
    'MANDI032': {'district': 'Saran', 'state': 'Bihar'},
    'MANDI017': {'district': 'Parbhani', 'state': 'Maharashtra'},
    'MANDI039': {'district': 'Anand', 'state': 'Gujarat'},
    
    # 4 mandis with missing state
    'MANDI057': {'district': 'Bareilly', 'state': 'Uttar Pradesh'},
    'MANDI054': {'district': 'Saharanpur', 'state': 'Uttar Pradesh'},
    'MANDI030': {'district': 'Sirsa', 'state': 'Haryana'},
    'MANDI002': {'district': 'Ludhiana', 'state': 'Punjab'}
}

In [ ]:
# Function to impute missing district and state
def fill_location(row):
    m_id = row['mandi_id']
    dist = row['district']
    st = row['state']
    
    if pd.isna(dist) and m_id in location_lookup:
        dist = location_lookup[m_id]['district']
    if pd.isna(st) and m_id in location_lookup:
        st = location_lookup[m_id]['state']
        
    return pd.Series([dist, st])


In [41]:
# Apply location imputation
df[['clean_district', 'clean_state']] = df.apply(fill_location, axis=1)

In [42]:
# AFTER 
print("AFTER Geographic Imputation Verification:- ")
print("Missing district entries after:", df['clean_district'].isnull().sum())
print("Missing state entries after:", df['clean_state'].isnull().sum())

AFTER Geographic Imputation Verification:- 
Missing district entries after: 0
Missing state entries after: 0
